# 07 — Core ICNN Lyapunov Experiments, Exp7-style

This notebook now follows the **same training pipeline as Exp7** for the main pendulum experiment:

1. load a good nominal `fhat` from Exp5,
2. generate rollout states using `fhat`,
3. create `random states + rollout states`,
4. freeze `fhat`,
5. train only the ICNN Lyapunov network `V`,
6. optimize `relu(gradV · fhat + alpha V)^2`,
7. evaluate whether the projection no longer destroys the good nominal dynamics.

The important comparison is:

```text
baseline MLP | fhat only | ICNN projected stable model
```

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start
    for p in [start, *start.parents]:
        if (p / "stable_icnn_physics").exists():
            return p
    raise RuntimeError("Start Jupyter from the DeepLearningFTN repository.")

REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT))

from stable_icnn_physics import BaselineDynamicsMLP, build_stable_model, make_system
from stable_icnn_physics.data import (
    dataset_base_name,
    generate_derivative_data,
    load_dataset,
    save_dataset,
    tensor_dataset,
)
from stable_icnn_physics.eval import (
    autoregressive_rollout_model,
    lyapunov_decrease_values,
    rollout_error,
    rollout_system,
)
from stable_icnn_physics.train import evaluate_derivative_mse
from stable_icnn_physics.rollout_augmented import (
    collect_rollout_states,
    train_lyapunov_v_only_on_states,
    projection_diagnostics,
)

torch.set_float32_matmul_precision("high")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CACHE_DIR = REPO_ROOT / "data" / "cache"
CKPT_DIR = REPO_ROOT / "checkpoints"
RESULTS_DIR = REPO_ROOT / "results"
PLOTS_DIR = RESULTS_DIR / "core_icnn_plots"
for d in [CACHE_DIR, CKPT_DIR, RESULTS_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEED = 0
TOLERANCE = 1e-5

print("repo:", REPO_ROOT)
print("device:", DEVICE, "| torch:", torch.__version__)

## Configuration

In [ ]:
# Main project system
SYSTEM_NAME = "damped_pendulum_4"
SYSTEM_KWARGS = {"friction": 0.3, "gravity": 9.81}
DT = 0.02
STEPS = 300

# Random derivative data used together with rollout states
TRAIN_SAMPLES = 50_000
TEST_SAMPLES = 10_000

# Exp7-style rollout-state augmentation
N_ROLLOUT_TRAIN_TRAJS = 500
N_ROLLOUT_VAL_TRAJS = 100
EPOCHS_V = 400
BATCH_V = 512

# Evaluation
EVAL_TRAJS = 16
EVAL_SEED = SEED + 123

# Architecture must match Exp5/Exp7 checkpoints
HIDDEN = 200
DEPTH = 3
LYAPUNOV_HIDDEN = 100
LYAPUNOV_EPS = 0.01
ALPHA = 1e-5
REHU_WIDTH = 0.01

# Checkpoints
PHASE1_STABLE_CKPT = CKPT_DIR / "p4_random_large_e500_alpha1e5_stable.pt"
BASELINE_CKPT = CKPT_DIR / "p4_random_large_e500_alpha1e5_baseline.pt"
ROLLOUT_AUG_CKPT = CKPT_DIR / "core_p4_exp7style_rollout_aug_stable.pt"

# Set True to retrain V. If False, loads ROLLOUT_AUG_CKPT if it exists.
RUN_V_TRAINING = True

print("phase1 stable exists:", PHASE1_STABLE_CKPT.exists(), PHASE1_STABLE_CKPT)
print("baseline exists:", BASELINE_CKPT.exists(), BASELINE_CKPT)
print("rollout-aug ckpt exists:", ROLLOUT_AUG_CKPT.exists(), ROLLOUT_AUG_CKPT)

## Load system and random derivative data

In [ ]:
system = make_system(SYSTEM_NAME, **SYSTEM_KWARGS)
state_dim = system.state_dim

for split, n in [("train", TRAIN_SAMPLES), ("test", TEST_SAMPLES)]:
    path = CACHE_DIR / dataset_base_name(
        system, split=split, n_samples=n, seed=SEED, dataset_type="derivative"
    )
    if not path.exists():
        print("generating", split, path.name)
        x, y = generate_derivative_data(system, n_samples=n, split=split, seed=SEED)
        save_dataset(path, x, y, metadata={"system": system.metadata(), "split": split})
    else:
        print("reusing", split, path.name)

train_path = CACHE_DIR / dataset_base_name(
    system, split="train", n_samples=TRAIN_SAMPLES, seed=SEED, dataset_type="derivative"
)
test_path = CACHE_DIR / dataset_base_name(
    system, split="test", n_samples=TEST_SAMPLES, seed=SEED, dataset_type="derivative"
)

x_train, y_train = load_dataset(train_path)
x_test, y_test = load_dataset(test_path)

train_ds = tensor_dataset(x_train, y_train)
test_ds = tensor_dataset(x_test, y_test)

print("system:", system.name, "state_dim:", state_dim)
print("random train:", x_train.shape, y_train.shape)
print("random test:", x_test.shape, y_test.shape)

## Load Exp5 nominal dynamics `fhat` and baseline

In [ ]:
def make_stable():
    return build_stable_model(
        dim=state_dim,
        hidden=HIDDEN,
        depth=DEPTH,
        lyapunov_hidden=LYAPUNOV_HIDDEN,
        lyapunov_eps=LYAPUNOV_EPS,
        alpha=ALPHA,
        rehu_width=REHU_WIDTH,
    )

def make_baseline():
    return BaselineDynamicsMLP(dim=state_dim, hidden=HIDDEN, depth=DEPTH)

if not PHASE1_STABLE_CKPT.exists():
    raise FileNotFoundError(f"Missing Exp5 phase-1 stable checkpoint: {PHASE1_STABLE_CKPT}")

phase1_stable = make_stable()
phase1_stable.load_state_dict(
    torch.load(PHASE1_STABLE_CKPT, map_location=DEVICE, weights_only=True)["model_state"]
)
phase1_stable.to(DEVICE).eval()

baseline = None
if BASELINE_CKPT.exists():
    baseline = make_baseline()
    baseline.load_state_dict(
        torch.load(BASELINE_CKPT, map_location=DEVICE, weights_only=True)["model_state"]
    )
    baseline.to(DEVICE).eval()
    print("baseline loaded")
else:
    print("baseline checkpoint missing; fhat/stable comparison will still run")

print("phase1 stable loaded")
print("fhat params:", sum(p.numel() for p in phase1_stable.fhat.parameters()))
print("V params:", sum(p.numel() for p in phase1_stable.V.parameters()))

## Generate rollout states using `fhat`

In [ ]:
if RUN_V_TRAINING:
    print("collecting fhat rollout states...")
    rollout_train_states = collect_rollout_states(
        phase1_stable.fhat,
        system,
        n_trajs=N_ROLLOUT_TRAIN_TRAJS,
        steps=STEPS,
        dt=DT,
        split="train",
        seed=SEED + 1,
        device=DEVICE,
    )
    rollout_val_states = collect_rollout_states(
        phase1_stable.fhat,
        system,
        n_trajs=N_ROLLOUT_VAL_TRAJS,
        steps=STEPS,
        dt=DT,
        split="test",
        seed=SEED + 2,
        device=DEVICE,
    )

    combined_train_states = np.concatenate([x_train, rollout_train_states], axis=0)
    combined_val_states = np.concatenate([x_test, rollout_val_states], axis=0)

    print("rollout train states:", rollout_train_states.shape)
    print("rollout val states:", rollout_val_states.shape)
    print("combined train states:", combined_train_states.shape)
    print("combined val states:", combined_val_states.shape)
else:
    print("Skipping rollout-state generation because RUN_V_TRAINING=False")

## Freeze `fhat` and train only `V`

In [ ]:
stable = make_stable()

if RUN_V_TRAINING:
    # Start from Exp5: good fhat + old V. Then replace only V through V-only training.
    stable.load_state_dict(
        torch.load(PHASE1_STABLE_CKPT, map_location=DEVICE, weights_only=True)["model_state"]
    )
    stable.to(DEVICE).eval()

    history = train_lyapunov_v_only_on_states(
        stable,
        combined_train_states,
        combined_val_states,
        epochs=EPOCHS_V,
        batch_size=BATCH_V,
        learning_rate=1e-3,
        eta_min=1e-4,
        device=DEVICE,
        print_every=40,
    )

    torch.save({"model_state": stable.state_dict(), "history": history.__dict__}, ROLLOUT_AUG_CKPT)
    print("saved:", ROLLOUT_AUG_CKPT)

    plt.figure(figsize=(8, 4))
    plt.semilogy(history.epoch, history.train_loss, label="train")
    plt.semilogy(history.epoch, history.val_loss, label="val")
    plt.semilogy(history.epoch, history.best_val_loss, label="best val", linestyle="--")
    plt.xlabel("epoch")
    plt.ylabel("V-only violation loss")
    plt.title("Exp7-style V-only rollout-augmented training")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    path = PLOTS_DIR / "core_p4_exp7style_vonly_training_loss.png"
    plt.savefig(path, dpi=160)
    plt.show()
    print("saved:", path)
else:
    if not ROLLOUT_AUG_CKPT.exists():
        raise FileNotFoundError(f"Missing {ROLLOUT_AUG_CKPT}; set RUN_V_TRAINING=True")
    stable.load_state_dict(
        torch.load(ROLLOUT_AUG_CKPT, map_location=DEVICE, weights_only=True)["model_state"]
    )
    stable.to(DEVICE).eval()
    history = None
    print("loaded:", ROLLOUT_AUG_CKPT)

## Evaluate: baseline vs fhat-only vs projected stable

In [ ]:
x0 = system.sample_initial_conditions(EVAL_TRAJS, split="test", seed=EVAL_SEED)
true_traj = rollout_system(system, x0, steps=STEPS, dt=DT)

stable_traj = autoregressive_rollout_model(
    stable, x0, steps=STEPS, dt=DT, device=DEVICE, wrap_fn=system.wrap_state
)
fhat_traj = autoregressive_rollout_model(
    stable.fhat, x0, steps=STEPS, dt=DT, device=DEVICE, wrap_fn=system.wrap_state
)
baseline_traj = None
if baseline is not None:
    baseline_traj = autoregressive_rollout_model(
        baseline, x0, steps=STEPS, dt=DT, device=DEVICE, wrap_fn=system.wrap_state
    )

err_stable = rollout_error(system, true_traj, stable_traj).mean(axis=1)
err_fhat = rollout_error(system, true_traj, fhat_traj).mean(axis=1)
err_baseline = None if baseline_traj is None else rollout_error(system, true_traj, baseline_traj).mean(axis=1)

dmse_stable = evaluate_derivative_mse(stable, test_ds, device=DEVICE)
dmse_fhat = evaluate_derivative_mse(stable.fhat, test_ds, device=DEVICE)
dmse_baseline = None if baseline is None else evaluate_derivative_mse(baseline, test_ds, device=DEVICE)

decrease = lyapunov_decrease_values(stable, x_test[:2048], device=DEVICE).ravel()
diag = projection_diagnostics(stable, stable_traj, device=DEVICE)

def summarize(err):
    return {"final": float(err[-1]), "mean": float(err.mean()), "max": float(err.max())}

summary = {
    "experiment": "core_p4_exp7style_rollout_augmented",
    "system": SYSTEM_NAME,
    "checkpoint": str(ROLLOUT_AUG_CKPT.relative_to(REPO_ROOT)),
    "phase1_checkpoint": str(PHASE1_STABLE_CKPT.relative_to(REPO_ROOT)),
    "baseline_checkpoint": str(BASELINE_CKPT.relative_to(REPO_ROOT)) if BASELINE_CKPT.exists() else None,
    "dt": DT,
    "steps": STEPS,
    "eval_trajs": EVAL_TRAJS,
    "derivative_mse_stable_projected": float(dmse_stable),
    "derivative_mse_fhat_only": float(dmse_fhat),
    "derivative_mse_baseline": None if dmse_baseline is None else float(dmse_baseline),
    "rollout_stable_projected": summarize(err_stable),
    "rollout_fhat_only": summarize(err_fhat),
    "rollout_baseline": None if err_baseline is None else summarize(err_baseline),
    "lyapunov_max_violation_projected": float(decrease.max()),
    "lyapunov_fraction_satisfied_projected": float(np.mean(decrease <= TOLERANCE)),
    "projection_fire_rate": diag["projection_fire_rate"],
    "correction_norm_mean": diag["correction_norm_mean"],
    "correction_norm_p95": diag["correction_norm_p95"],
    "correction_norm_max": diag["correction_norm_max"],
    "nominal_violation_mean": diag["nominal_violation_mean"],
    "nominal_violation_max": diag["nominal_violation_max"],
    "discrete_dV_frac_nonpositive": diag["discrete_dV_frac_nonpositive"],
    "discrete_dV_max": diag["discrete_dV_max"],
}

summary_path = RESULTS_DIR / "core_p4_exp7style_rollout_augmented_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))
print("saved:", summary_path)

## Rollout error plots

In [ ]:
t = np.arange(STEPS + 1) * DT

plt.figure(figsize=(8, 4))
if err_baseline is not None:
    plt.plot(t, err_baseline, label="baseline MLP")
plt.plot(t, err_fhat, label="fhat only")
plt.plot(t, err_stable, label="ICNN projected")
plt.xlabel("time [s]")
plt.ylabel("mean state error")
plt.title("Pendulum-4 rollout error — Exp7-style training")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
path = PLOTS_DIR / "core_p4_exp7style_rollout_error.png"
plt.savefig(path, dpi=160)
plt.show()
print("saved:", path)

plt.figure(figsize=(8, 4))
if err_baseline is not None:
    plt.semilogy(t, err_baseline + 1e-12, label="baseline MLP")
plt.semilogy(t, err_fhat + 1e-12, label="fhat only")
plt.semilogy(t, err_stable + 1e-12, label="ICNN projected")
plt.xlabel("time [s]")
plt.ylabel("mean state error, log scale")
plt.title("Pendulum-4 rollout error log-scale — Exp7-style training")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
path = PLOTS_DIR / "core_p4_exp7style_rollout_error_log.png"
plt.savefig(path, dpi=160)
plt.show()
print("saved:", path)

## Projection diagnostics

In [ ]:
t_mid = t[:-1]

plt.figure(figsize=(8, 4))
plt.plot(t_mid, diag["fire"].mean(axis=1))
plt.xlabel("time [s]")
plt.ylabel("fraction of trajectories")
plt.ylim(-0.02, 1.02)
plt.title("Projection fire rate over rollout")
plt.grid(True, alpha=0.3)
plt.tight_layout()
path = PLOTS_DIR / "core_p4_exp7style_projection_fire_rate.png"
plt.savefig(path, dpi=160)
plt.show()
print("saved:", path)

plt.figure(figsize=(8, 4))
plt.plot(t_mid, diag["correction_norm"].mean(axis=1), label="mean")
plt.plot(t_mid, np.quantile(diag["correction_norm"], 0.95, axis=1), label="p95")
plt.xlabel("time [s]")
plt.ylabel("|| correction ||")
plt.title("Projection correction magnitude")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
path = PLOTS_DIR / "core_p4_exp7style_correction_norm.png"
plt.savefig(path, dpi=160)
plt.show()
print("saved:", path)

plt.figure(figsize=(8, 4))
plt.hist(diag["violation"].ravel(), bins=80)
plt.axvline(0, linestyle="--")
plt.xlabel("gradV · fhat + alpha V")
plt.ylabel("count")
plt.title("Nominal Lyapunov violation distribution")
plt.grid(True, alpha=0.3)
plt.tight_layout()
path = PLOTS_DIR / "core_p4_exp7style_nominal_violation_hist.png"
plt.savefig(path, dpi=160)
plt.show()
print("saved:", path)

## Example state trajectories

In [ ]:
coord_names = [f"theta_{i}" for i in range(4)] + [f"omega_{i}" for i in range(4)]
traj_id = 0

for j, name in enumerate(coord_names):
    plt.figure(figsize=(8, 3.5))
    plt.plot(t, true_traj[:, traj_id, j], label="true")
    if baseline_traj is not None:
        plt.plot(t, baseline_traj[:, traj_id, j], label="baseline")
    plt.plot(t, fhat_traj[:, traj_id, j], label="fhat only")
    plt.plot(t, stable_traj[:, traj_id, j], label="ICNN projected")
    plt.xlabel("time [s]")
    plt.ylabel(name)
    plt.title(f"Pendulum-4 coordinate: {name}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    path = PLOTS_DIR / f"core_p4_exp7style_traj_{j}_{name}.png"
    plt.savefig(path, dpi=160)
    plt.show()

## Interpretation

This notebook is the version that should support the final report's strongest claim:

> The best result is obtained by separating nominal dynamics learning from Lyapunov learning.
> A good `fhat` is learned first; then `fhat` is frozen and the ICNN Lyapunov function is trained on random + rollout states.

This directly addresses the distribution shift between random training points and autoregressive rollout states.